# 7교시. 틀린 값을 걸러 CSV로 저장하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/07_validation_export.ipynb)

**목표:** 필수값과 품목 합계를 확인하고 CSV를 만듭니다.

**결과물:** `receipt.csv`

- 기본 경로는 API 키와 OCR 모델 다운로드가 필요 없습니다.
- 선택 실습은 기본값이 `False`입니다.
- 수업이 지정한 공개 실물·합성 샘플만 사용합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
import csv
import io
from copy import deepcopy

SAMPLE_RECEIPT = {'document_type': 'receipt', 'store_name': '샘플문구점', 'date': '2026-07-27', 'total_amount': 5000, 'items': [{'name': '연필', 'quantity': 2, 'unit_price': 1000, 'line_total': 2000}, {'name': '노트', 'quantity': 1, 'unit_price': 3000, 'line_total': 3000}], 'source_mode': 'mock'}
MISSING_STORE = deepcopy(SAMPLE_RECEIPT)
MISSING_STORE["store_name"] = None
WRONG_TOTAL = deepcopy(SAMPLE_RECEIPT)
WRONG_TOTAL["total_amount"] = 6000


## 핵심 3개

1. 검증 결과는 valid·warnings·errors로 나눕니다.
2. 자료형과 업무 규칙은 다른 검사입니다.
3. 품목 하나가 CSV의 한 행이 됩니다.


In [ ]:
def validate_receipt(data):
    errors = []

    for field in ("store_name", "date", "total_amount", "items"):
        if data.get(field) in (None, "", []):
            errors.append(f"필수값 누락: {field}")

    item_sum = sum(
        item["line_total"] for item in data.get("items", [])
    )
    if data.get("total_amount") != item_sum:
        errors.append("품목 합계와 총액이 다릅니다.")

    return {
        "valid": not errors,
        "warnings": [],
        "errors": errors,
    }


## 실습. 세 데이터 검증


In [ ]:
normal = validate_receipt(SAMPLE_RECEIPT)
missing = validate_receipt(MISSING_STORE)
wrong_total = validate_receipt(WRONG_TOTAL)

assert normal["valid"]
assert not missing["valid"]
assert not wrong_total["valid"]

print("정상:", normal)
print("누락:", missing)
print("합계 불일치:", wrong_total)


In [ ]:
def safe_text(value):
    if isinstance(value, str) and value.startswith(("=", "+", "-", "@")):
        return "'" + value
    return value


def receipt_rows(data):
    rows = []
    for item in data["items"]:
        row = {
            "store_name": data["store_name"],
            "date": data["date"],
            "total_amount": data["total_amount"],
            "item_name": item["name"],
            "quantity": item["quantity"],
            "unit_price": item["unit_price"],
            "line_total": item["line_total"],
        }
        rows.append({key: safe_text(value) for key, value in row.items()})
    return rows


columns = [
    "store_name", "date", "total_amount", "item_name",
    "quantity", "unit_price", "line_total",
]
output_path = OUTPUT_DIR / "receipt.csv"
with output_path.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows(receipt_rows(SAMPLE_RECEIPT))

print("저장 완료:", output_path)
print(output_path.read_text(encoding="utf-8-sig"))


In [ ]:
RUN_COLAB_DOWNLOAD = False

if RUN_COLAB_DOWNLOAD:
    from google.colab import files
    files.download(str(output_path))
else:
    print("자동 다운로드를 건너뛰었습니다. Colab 파일 영역에서 받을 수 있습니다.")


## mock 대체 경로

이전 앱 없이 내장 `SAMPLE_RECEIPT`를 같은 검증·CSV 함수에 전달합니다.
